## SuperString
---

In [2]:
/*
secrecy::SecretString handles zeroize and display, no mlock, no mprotect, no encryption.
memsecurity::EncryptedMem is either too far beyond poorly maintained to take seriously.
*/

In [3]:
:dep chacha20poly1305 = "0.10.1"
:dep getrandom = "0.3.4"
:dep rustix = { version = "1.1.3", features = ["mm", "param"] }

In [4]:
use rustix::mm::{mlock, mprotect, munlock, MprotectFlags};
use rustix::param::page_size;
use std::{
	alloc::{alloc, dealloc, Layout},
	ffi::c_void,
	fmt,
	ptr::{copy_nonoverlapping, write_volatile, NonNull},
	sync::atomic::{compiler_fence, Ordering},
};


In [5]:
const KEY_SIZE: usize = 32;
const NONCE_SIZE: usize = 24;
const HEADER_SIZE: usize = KEY_SIZE + NONCE_SIZE;

In [6]:
fn encrypt(msg: Vec<u8>, key: [u8; KEY_SIZE], nonce: [u8; NONCE_SIZE]) -> Vec<u8> {
	use chacha20poly1305::{
		aead::{Aead, KeyInit},
		XChaCha20Poly1305,
	};
	let cipher = XChaCha20Poly1305::new(&key.into());
	cipher.encrypt(&nonce.into(), &msg[..]).unwrap()
}

fn decrypt(msg_enc: Vec<u8>, key: [u8; KEY_SIZE], nonce: [u8; NONCE_SIZE]) -> Option<Vec<u8>> {
	use chacha20poly1305::{
		aead::{Aead, KeyInit},
		XChaCha20Poly1305,
	};
	let cipher = XChaCha20Poly1305::new(&key.into());
	cipher.decrypt(&nonce.into(), msg_enc.as_ref()).ok()
}

In [7]:
pub struct SuperString {
	page: NonNull<u8>,
	secret_offset: usize,
	encrypted_len: usize,
}

In [8]:
impl Drop for SuperString {
	fn drop(&mut self) {
		unsafe {
			let page_addr = self.page.as_ptr();
			let void_addr = page_addr as *mut c_void;
			let page_sz = page_size();
			let _ = mprotect(void_addr, page_sz, MprotectFlags::WRITE);
			for i in 0..page_sz {
				write_volatile(page_addr.add(i), 0);
			}
			compiler_fence(Ordering::SeqCst);
			let _ = munlock(void_addr, page_sz);
			let layout = Layout::from_size_align_unchecked(page_sz, page_sz);
			dealloc(page_addr, layout);
		}
	}
}

In [9]:
impl SuperString {
	pub fn new(source: String) -> Self {
		let page_sz = page_size();
		let mut key = [0u8; KEY_SIZE];
		let mut nonce = [0u8; NONCE_SIZE];
		getrandom::fill(&mut key).expect("Failed to generate random key");
		getrandom::fill(&mut nonce).expect("Failed to generate random nonce");
		let encrypted = encrypt(source.as_bytes().to_vec(), key, nonce);
		let encrypted_len = encrypted.len();
		let mut source = source;
		unsafe {
			let bytes = source.as_bytes_mut();
			for i in 0..bytes.len() {
				write_volatile(bytes.as_mut_ptr().add(i), 0);
			}
			compiler_fence(Ordering::SeqCst);
		}
		drop(source);
		// Layout: [32-byte key][24-byte nonce][encrypted data][padding]
		if HEADER_SIZE + encrypted_len > page_sz {
			panic!("Secret too large for single page");
		}
		unsafe {
			let layout = Layout::from_size_align(page_sz, page_sz).expect("Invalid layout");
			let page_addr = alloc(layout);
			if page_addr.is_null() {
				panic!("Allocation failed");
			}
			copy_nonoverlapping(key.as_ptr(), page_addr, KEY_SIZE);
			copy_nonoverlapping(nonce.as_ptr(), page_addr.add(KEY_SIZE), NONCE_SIZE);
			let secret_addr = page_addr.add(HEADER_SIZE);
			copy_nonoverlapping(encrypted.as_ptr(), secret_addr, encrypted_len);
			let void_addr = page_addr as *mut c_void;
			mlock(void_addr, page_sz).expect("mlock failed");
			mprotect(void_addr, page_sz, MprotectFlags::empty()).expect("mprotect failed");
			Self {
				page: NonNull::new(page_addr).unwrap(),
				secret_offset: HEADER_SIZE,
				encrypted_len,
			}
		}
	}

	pub fn use_secret<F, R>(&self, func: F) -> R
	where
		F: FnOnce(&str) -> R,
	{
		unsafe {
			let page_addr = self.page.as_ptr();
			let void_addr = page_addr as *mut c_void;
			let page_sz = page_size();
			mprotect(void_addr, page_sz, MprotectFlags::READ).unwrap();
			let mut key = [0u8; KEY_SIZE];
			let mut nonce = [0u8; NONCE_SIZE];
			copy_nonoverlapping(page_addr, key.as_mut_ptr(), KEY_SIZE);
			copy_nonoverlapping(page_addr.add(KEY_SIZE), nonce.as_mut_ptr(), NONCE_SIZE);
			let secret_addr = page_addr.add(self.secret_offset);
			let encrypted = std::slice::from_raw_parts(secret_addr, self.encrypted_len).to_vec();
			let decrypted = decrypt(encrypted, key, nonce).expect("Decryption failed");
			let secret_str = std::str::from_utf8_unchecked(&decrypted);
			let res = func(secret_str);
			mprotect(void_addr, page_sz, MprotectFlags::empty()).unwrap();
			res
		}
	}

	pub fn expose_secret(&self) -> String {
		self.use_secret(|sec| sec.to_string())
	}
}

impl fmt::Debug for SuperString {
	fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
		write!(f, "SuperString(<REDACTED>)")
	}
}

impl fmt::Display for SuperString {
	fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
		write!(f, "********")
	}
}

In [10]:
{
    let q = SuperString::new("test".into());
    println!("{:?}", q.expose_secret());
};

"test"
